# 10 — Dataset Assembly

Merges all training sources into one clean, balanced dataset ready for fine-tuning.

**Why merge?** Notebooks 03–07b accumulate hundreds of high-quality, curated pairs
in `knowledge_pairs.jsonl`. Notebook 08 generates hundreds more validated synthetic
samples. Without this merge step, the final fine-tune (notebook 11) would see only
the synthetic data — and all the curated pairs would go to waste.

**What this notebook does:**
1. Load validated synthetic samples from notebook 09
2. Load all curated knowledge pairs from notebooks 03, 05, 06, 07, and 07b
3. Convert everything to ChatML format with the canonical system prompt
4. Deduplicate by instruction prefix
5. Cap per-task type at 40% of total (prevents any one task from dominating)
6. Shuffle and split into train / valid / test (90 / 5 / 5)
7. Write `mlx/` format ready for `mlx_lm lora --use-chat-template`

The system prompt is built from `knowledge.json` so it always lists all 61 actions —
consistent with the prompts used during warm-start and synthetic generation.

**Inputs:**
- `../data/04_validated/valid_samples.jsonl` — validated synthetic samples (NB09)
- `../data/02_knowledge/knowledge_pairs.jsonl` — curated pairs (NB03+05+06+07+07b)

**Output:** `../data/05_dataset/train.jsonl`  (full format with metadata)
            `../data/05_dataset/valid.jsonl`
            `../data/05_dataset/test.jsonl`
            `../data/05_dataset/mlx/train.jsonl`  (messages-only, for mlx_lm lora)
            `../data/05_dataset/mlx/valid.jsonl`
            `../data/05_dataset/stats.json`

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

import sys, importlib
_cfg_dir = SCRIPT_DIR
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

DATA_IN  = DATA_ROOT / '04_validated'
DATA_OUT = DATA_ROOT / '05_dataset'
MLX_DIR  = DATA_OUT / 'mlx'
DATA_OUT.mkdir(parents=True, exist_ok=True)
MLX_DIR.mkdir(parents=True, exist_ok=True)

# ── Load validated synthetic samples (notebook 09 output) ────────────────────
samples = []
valid_samples_file = DATA_IN / 'valid_samples.jsonl'
if not valid_samples_file.exists():
    raise FileNotFoundError(
        f'{valid_samples_file} not found — run notebook 09 first.'
    )
with open(valid_samples_file) as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f'Loaded {len(samples)} validated synthetic samples')
print('Distribution:', dict(Counter(s['task_type'] for s in samples)))

In [ ]:
# ── Load curated knowledge pairs (notebooks 03+05+06+07+07b output) ──────────
# These pairs are already high-quality (exec-validated or doc-grounded).
# They are not in the synthetic samples file, so they would be dropped without
# this merge step — meaning all the effort of NB05/06/07/07b would be wasted.

def _source_to_task_type(source):
    """Map knowledge_pairs source tags to task_type used in this notebook."""
    src = source.lower()
    if any(src.startswith(p) for p in ('book_qa:', 'wiki:', 'actions_explain', 'actions_which')):
        return 'syntax_qa'
    if src.startswith('repair'):
        return 'debugging'
    # example:*, mutation, recombination, spec_to_code, readme_to_code,
    # actions_usage, actions_alias, actions_context, book, aro_by_example, proposal:*
    return 'code_generation'


kp_added = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            p = json.loads(line)
            instruction = p.get('instruction', '').strip()
            output      = p.get('output', '').strip()
            if not instruction or not output:
                continue
            source    = p.get('source', 'knowledge_pairs')
            task_type = _source_to_task_type(source)
            samples.append({
                'task_type':  task_type,
                'instruction': instruction,
                'output':      output,
                'source':      source,
                'score':       p.get('score', 1.0),
                'valid':       True,   # already curated
            })
            kp_added += 1
else:
    print(f'WARNING: {PAIRS_FILE} not found — run notebooks 03, 05, 06, 07, 07b first.')

print(f'Added {kp_added} curated knowledge pairs')
print(f'Total samples before dedup: {len(samples)}')
print('Full distribution:', dict(Counter(s['task_type'] for s in samples)))

## System prompt

In [ ]:
# Load knowledge base and build the canonical system prompt.
# Uses build_system_prompt(kb) from config so NB10 uses the same prompt
# as NB08 and the warm-start NB04 (consistent system token across all training).
with open(KB_FILE) as _kb_f:
    _kb = json.load(_kb_f)

ARO_SYSTEM_PROMPT = build_system_prompt(_kb)
print(f'System prompt: {len(ARO_SYSTEM_PROMPT)} chars  ({len(_kb["actions"])} actions referenced)')

## Convert to ChatML

In [ ]:
def build_messages(sample):
    task  = sample.get('task_type', '')
    instr = sample.get('instruction', '').strip()
    inp   = sample.get('input', '').strip()
    out   = sample.get('output', '').strip()

    if not out:
        return None

    if task == 'fim':
        prefix = sample.get('prefix', '')
        suffix = sample.get('suffix', '')
        middle = sample.get('middle', '')
        if not middle.strip():
            return None
        user = f'Complete the missing ARO statement(s). Only output the missing line(s).\n\n```aro\n{prefix}\n<FILL_HERE>\n{suffix}\n```'
        return user, middle

    if inp:
        user = f'{instr}\n\n```aro\n{inp}\n```'
    else:
        user = instr

    return (user, out) if user.strip() else None


def to_chatml(sample):
    result = build_messages(sample)
    if result is None:
        return None
    user_content, assistant_content = result
    if len(user_content) < 10 or len(assistant_content) < 10:
        return None
    return {
        'messages': [
            {'role': 'system',    'content': ARO_SYSTEM_PROMPT},
            {'role': 'user',      'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ],
        'task_type': sample.get('task_type', ''),
        'source':    sample.get('source', sample.get('domain', sample.get('scenario', ''))),
    }


chatml = [to_chatml(s) for s in samples]
chatml = [c for c in chatml if c is not None]
print(f'Converted: {len(chatml)}  (dropped {len(samples) - len(chatml)} empty)')

## Deduplicate

In [ ]:
seen, deduped = set(), []
for s in chatml:
    key = s['messages'][1]['content'][:300]
    if key not in seen:
        seen.add(key)
        deduped.append(s)
print(f'After dedup: {len(deduped)}')

## Cap per-task count (max 40% per type)

In [ ]:
MAX_PER_TYPE = max(500, len(deduped) // 3)
capped, type_counts = [], Counter()
for s in deduped:
    t = s['task_type']
    if type_counts[t] < MAX_PER_TYPE:
        capped.append(s)
        type_counts[t] += 1

print(f'After cap: {len(capped)}')
for t, n in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {t:25s}: {n}')

## Train / valid / test split  (90 / 5 / 5)

In [ ]:
random.seed(42)
random.shuffle(capped)
n     = len(capped)
t_end = int(n * 0.90)
v_end = int(n * 0.95)
train = capped[:t_end]
valid = capped[t_end:v_end]
test  = capped[v_end:]
print(f'train={len(train)}  valid={len(valid)}  test={len(test)}')

## Save full format + mlx format

In [ ]:
def save_jsonl(data, path):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    print(f'  {path.name}: {len(data)} samples')


# Full format (with task_type / source metadata)
save_jsonl(train, DATA_OUT / 'train.jsonl')
save_jsonl(valid, DATA_OUT / 'valid.jsonl')   # NOTE: valid (not val) — mlx-lm convention
save_jsonl(test,  DATA_OUT / 'test.jsonl')

# mlx-lm format: only the "messages" key (mlx_lm.lora --use-chat-template reads this)
save_jsonl([{'messages': s['messages']} for s in train], MLX_DIR / 'train.jsonl')
save_jsonl([{'messages': s['messages']} for s in valid], MLX_DIR / 'valid.jsonl')

# Stats
stats = {
    'total': len(capped), 'train': len(train), 'valid': len(valid), 'test': len(test),
    'task_counts': dict(type_counts),
    'avg_user_len':      int(sum(len(s['messages'][1]['content']) for s in capped) / max(1, len(capped))),
    'avg_assistant_len': int(sum(len(s['messages'][2]['content']) for s in capped) / max(1, len(capped))),
}
with open(DATA_OUT / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'\nstats.json: avg user={stats["avg_user_len"]} chars, avg assistant={stats["avg_assistant_len"]} chars')

## Sanity check

In [ ]:
for s in random.sample(train, 3):
    print('─' * 60)
    print(f'Task: {s["task_type"]}')
    print(f'User:      {s["messages"][1]["content"][:120]}...')
    print(f'Assistant: {s["messages"][2]["content"][:120]}...')
print('─' * 60)